# Community Identification

In [2]:
import sys

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["DGLBACKEND"] = "pytorch"

import torch

DEVICE = torch.device("cuda:0")  # GPU 1 becomes cuda:0 now

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
(12, 0)


In [3]:

import torch
from collections import defaultdict
import statistics as stats
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
 
from dotenv import load_dotenv
from huggingface_hub import login
 
load_dotenv()
token = os.getenv("HF_TOKEN")
login(token=token)
 
# MODEL_ID = "Qwen/Qwen3-14B"
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
# MODEL_ID = "mistralai--Mistral-7B-v0.1"
# MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"   # gated -- must accept license + login() above
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",       # spreads layers across available GPU(s), offloads to CPU if needed
    torch_dtype=torch.bfloat16,
)
model.eval()
print("Model loaded.")

/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   0%|          | 2/579 [00:00<01:15,  7.61it/s]/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 579/579 [00:02<00:00, 203.30it/s]


Model loaded.


## 1. Load per-subject structure prompts

In [4]:

import os
import json
import re
import torch
import pandas as pd

  
ROI_NAMES_JSON = "../aal_roi_names.json"                # region-name legend, matches BLEG's "{template}" field
META_CSV = "../subject_summary.csv"


EDGE_LIST_NUMBERS = "../edge_list_top10pos_numbers.json"   # {subject_id: "Node feature:\n...\n\nEdge feature:\n..."}


EDGE_LIST_NAMES = "../edge_list_top10pos_names.json"  # {subject_id: "Node feature:\n...\n\nEdge feature:\n..."}
EDGE_LIST_NAMES_ONLY = "../edge_only_from_top10pos_names.json"

ZSCORED_NUMBERS = "../zscore_top10pos_numbers.json"  
ZSCORED_NAMES = "../zscore_top10pos_names.json" 
ROI_NAMES_JSON = "../aal_roi_names.json"                # region-name legend, matches BLEG's "{template}" field
META_CSV = "../subject_summary.csv"
DATASET_NAME = "ADHD-200"
LABEL1, LABEL2 = "Control", "ADHD"
HUB_GROUND_TRUTH = "../hub_ground_truth_20roi.json"  # {subject_id: "Control" or "ADHD"}
COMMUNITY_GROUND_TRUTH = "../community_ground_truth.json"
COMMUNITY_GROUND_TRUTH_COUNTS = "../community_ground_truth_counts.json"
ROI_NETWORK_JSON = "../roi_network.json"

PREPROCESS_TEMPLATE = "AAL116"
TASK_DESC = "interpreting functional connectivity patterns and reasoning about the most likely diagnostic group"
N_SUBJECTS = 768
MAX_NEW_TOKENS = 800

In [5]:
# with open(EDGE_LIST_NUMBERS) as f:
# with open(EDGE_LIST_NAMES) as f:
with open(EDGE_LIST_NAMES_ONLY) as f:
# with open(COMMUNITY_GROUND_TRUTH_COUNTS) as f:
# with open(COMMUNITY_GROUND_TRUTH) as f:
# with open(HUB_GROUND_TRUTH) as f:
# with open(ZSCORED_NUMBERS) as f:
# with open(ZSCORED_NAMES) as f:
    structure_prompts_all = json.load(f)

with open("../network_names.json") as f:
    NETWORK_NAMES = json.load(f)

if os.path.exists(ROI_NAMES_JSON):
    with open(ROI_NAMES_JSON) as f:
        roi_names = json.load(f)
    print(f"Loaded {len(roi_names)} AAL116 ROI names")
else:
    roi_names = None
    print("[warn] aal_roi_names.json not found -- description will omit the ROI-name legend")

if os.path.exists(ROI_NETWORK_JSON):
    with open(ROI_NETWORK_JSON) as f:
        roi_network = json.load(f)
    roi_to_network = dict(zip(roi_names, roi_network))
    print(f"Loaded roi_to_network for {len(roi_to_network)} ROIs")
else:
    roi_to_network = None
    print("[warn] roi_network.json not found -- community prompt will omit network labels")
# 1. Length/order sanity check
assert len(roi_names) == len(roi_network), (
    f"Length mismatch: roi_names={len(roi_names)}, roi_network={len(roi_network)} -- "
    "these two files were likely generated in different runs and are NOT safe to zip()."
)

# 2. Coverage check against the REAL edge-list data -- catches any region name
# that .get(a, "?") would otherwise silently swallow with no error.
all_edge_regions = set()
for text in structure_prompts_all.values():
    for line in text.strip().split("\n"):
        parts = line.split(":")
        if len(parts) == 3:
            all_edge_regions.add(parts[0])
            all_edge_regions.add(parts[2])

missing = all_edge_regions - set(roi_to_network.keys())
if missing:
    print(f"\n⚠️ {len(missing)} region names appear in your edge data but have NO match "
          f"in roi_to_network -- these will silently show as '?' in every prompt:")
    print(sorted(missing))
else:
    print(f"\n✅ All {len(all_edge_regions)} region names in the edge data have a valid network mapping.")


if os.path.exists(META_CSV):
    meta = pd.read_csv(META_CSV, dtype={"subject_id": str})
else:
    meta = None

subject_ids = list(structure_prompts_all.keys())[:N_SUBJECTS]
print(f"Loaded {len(structure_prompts_all)} subjects, running on first {len(subject_ids)}")
brain_graph_texts = dict(structure_prompts_all)
print("\nExample BrainGraph text (subject 1):")
print(brain_graph_texts[subject_ids[5]])

Loaded 116 AAL116 ROI names
Loaded roi_to_network for 116 ROIs

✅ All 116 region names in the edge data have a valid network mapping.
Loaded 768 subjects, running on first 768

Example BrainGraph text (subject 1):
Rectus_L:0.92:Rectus_R
Lingual_L:0.91:Lingual_R
Frontal_Sup_Orb_R:0.90:Rectus_R
Parietal_Sup_L:0.88:Parietal_Sup_R
Paracentral_Lobule_L:0.88:Paracentral_Lobule_R
Frontal_Med_Orb_R:0.87:Rectus_R
Cerebelum_Crus2_L:0.87:Cerebelum_Crus2_R
Cingulum_Ant_L:0.87:Cingulum_Ant_R
Cerebelum_4_5_L:0.86:Cerebelum_4_5_R
Frontal_Med_Orb_L:0.86:Frontal_Med_Orb_R
Cingulum_Mid_L:0.85:Cingulum_Mid_R
Cingulum_Post_L:0.83:Cingulum_Post_R
Frontal_Sup_Medial_L:0.83:Frontal_Sup_Medial_R
Cerebelum_8_L:0.83:Cerebelum_8_R
Insula_R:0.83:Putamen_R
Cerebelum_9_L:0.82:Cerebelum_9_R
Olfactory_L:0.81:Olfactory_R
Frontal_Sup_Orb_R:0.81:Rectus_L
Cerebelum_7b_L:0.81:Cerebelum_8_L
Cerebelum_3_L:0.81:Vermis_1_2
ParaHippocampal_L:0.81:ParaHippocampal_R
Cuneus_R:0.80:Occipital_Sup_L
Supp_Motor_Area_L:0.80:Supp_Motor

In [6]:
import json
import os

# Define the file path
file_path = "../community_ground_truth_counts.json"

def check_community_counts(path):
    if not os.path.exists(path):
        print(f"Error: File not found at {path}")
        return

    with open(path, 'r') as f:
        data = json.load(f)

    all_valid = True
    mismatches = []

    print(f"Checking community structures in {path}...\n")

    for subject_id, content in data.items():
        # Get counts
        within_counts = len(content.get("within_network", {}))
        between_counts = len(content.get("between_network", {}))
        total = within_counts + between_counts
        
        # Validation logic (8 within, 28 between)
        if within_counts != 8 or between_counts != 28:
            all_valid = False
            mismatches.append({
                "id": subject_id,
                "within": within_counts,
                "between": between_counts,
                "total": total
            })

    if all_valid:
        print(f"✅ Success! All {len(data)} subjects have 8 within-network and 28 between-network entries (Total 36).")
    else:
        print(f"❌ Found {len(mismatches)} subjects with incorrect community counts:")
        for m in mismatches:
            print(f" - Subject {m['id']}: Within={m['within']}, Between={m['between']} (Total {m['total']})")

# Run the check
check_community_counts(file_path)

Checking community structures in ../community_ground_truth_counts.json...

✅ Success! All 768 subjects have 8 within-network and 28 between-network entries (Total 36).


## 2. Prompt Templates -- Hub Identification (GraphArena-style, 1-shot CoT) + Classification (independent prompt, no hub info passed in)

In [ ]:
TOP_K_PREDICTED = 10
TOP_K_GROUND_TRUTH = 10
RETRY_ATTEMPTS = 2

THINK_BUDGET = 200     # kept low, per your request
ANSWER_BUDGET = 300    # dedicated space for the actual JSON/list, unaffected by thinking length

MAX_NEW_TOKENS_COMMUNITY = 400 # or 1500
MAX_NEW_TOKENS_CLS = 400 # or 500


In [9]:

COMMUNITY_EXAMPLE = """**Example**
- Regions and their networks: Frontal_Sup_L (Cont), Frontal_Sup_R (Cont), Parietal_Inf_L (DorsAttn), Occipital_Mid_L (Vis), Temporal_Sup_R (SalVentAttn)
- Functional connections: Frontal_Sup_L to Parietal_Inf_L, Frontal_Sup_R to Parietal_Inf_L, Frontal_Sup_L to Occipital_Mid_L, Frontal_Sup_L to Temporal_Sup_R, Parietal_Inf_L to Occipital_Mid_L, Frontal_Sup_L to Frontal_Sup_R

In this network, the regions belong to four functional networks: Cont (Frontal_Sup_L, Frontal_Sup_R), DorsAttn (Parietal_Inf_L), Vis (Occipital_Mid_L), and SalVentAttn (Temporal_Sup_R). Counting connections for each network pair (same network on both ends = within-network, different networks = between-network): Frontal_Sup_L to Parietal_Inf_L and Frontal_Sup_R to Parietal_Inf_L are both Cont-DorsAttn, so Cont-DorsAttn appears 2 times. Frontal_Sup_L to Occipital_Mid_L is Cont-Vis (1 time). Frontal_Sup_L to Temporal_Sup_R is Cont-SalVentAttn (1 time). Parietal_Inf_L to Occipital_Mid_L is DorsAttn-Vis (1 time). Frontal_Sup_L to Frontal_Sup_R is Cont-Cont, a within-network connection (1 time). Ranking all network pairs by connection count: the network pair with the highest number of connections is Cont-DorsAttn with 2 connections. Therefore, the top 5 network pairs are [Cont-DorsAttn, Cont-Vis, Cont-SalVentAttn, DorsAttn-Vis, Cont-Cont], where Cont-Cont is the only within-network pair among them.
"""

def reformat_edges_with_networks(raw_edge_text, roi_to_network):
    """Same role as reformat_edges_to_graph_arena_style, but attaches each
    region's Yeo-7 label so the model can group by network."""
    lines = [ln.strip() for ln in raw_edge_text.strip().split("\n") if ln.strip()]
    pairs = []
    for line in lines:
        parts = line.split(":")
        if len(parts) == 3:
            a, _w, b = parts
            net_a = roi_to_network.get(a, "?") if roi_to_network else "?"
            net_b = roi_to_network.get(b, "?") if roi_to_network else "?"
            pairs.append(f"{a} ({net_a}) to {b} ({net_b})")   # no skip
        else:
            pairs.append(line)
    return ", ".join(pairs)


def build_community_prompt(brain_graph_text, roi_names, roi_to_network, top_k=TOP_K_PREDICTED):
        region_list_str = (
                ", ".join(f"{r} ({roi_to_network.get(r, '?')})" for r in roi_names)
                if roi_names and roi_to_network else "(region legend unavailable)"
            )
        edges_str = reformat_edges_with_networks(brain_graph_text, roi_to_network)

        return f"""You are required to identify the most densely connected functional network pairs in the given brain functional connectivity network and output the top pairs ranked by connectivity.
Each region belongs to one of 8 functional groupings: the 7 Yeo networks (Vis, SomMot, DorsAttn, SalVentAttn, Limbic, Cont, Default) plus 'Unassigned' for regions with no cortical Yeo-7 overlap. A network pair (same network twice for within-network, or two different networks for between-network) is highly connected if its regions appear frequently together across the connection list. To identify the top pairs, count how many times each network pair appears across the connection list -- pairs appearing most frequently are the most connected.

{COMMUNITY_EXAMPLE}
**Problem to Solve**
- Regions and their networks: {region_list_str}
- Functional connections: {edges_str}

Identify the top {top_k} network pairs in this network, ranked from most to least connected. Present your answer in the following format: [Pair1, Pair2, Pair3, ..., Pair{top_k}]"""



In [10]:
def build_classification_prompt(brain_graph_text, community_pairs):
    community_line = ""
    if community_pairs:
        community_str = ", ".join(community_pairs)
        community_line = f"\nIdentified densely connected network pairs (highest connectivity): {community_str}\n"

    return f"""Template: This data comes from the {DATASET_NAME} dataset, preprocessed using the {PREPROCESS_TEMPLATE} brain atlas template (116 regions of interest).
Description: Functional connectivity edges for one subject from resting-state fMRI. Format: RegionA:weight:RegionB, identified by their anatomical region name (AAL atlas).
Task: Based on the edge list below{" and the identified network pairs" if community_pairs else ""}, predict Control or ADHD with a confidence score for each class.
Request: Output only this JSON:
{{
"prediction": "{LABEL1} or {LABEL2}",
"class_confidence": {{
"{LABEL1}": 0.0,
"{LABEL2}": 0.0
}},
"reasoning": "(1-2 sentence)"
}}
Edges:
{brain_graph_text}
{community_line}"""

In [ ]:
## 5. Response parsers
def parse_community_list(raw):
    """
    Extracts a bracketed list [Region1, Region2, ...] from the model's
    free-text response. Returns [] if no valid bracket list is found
    (treated as 'missing' downstream).
    """
    raw = raw.strip()
    match = re.search(r"\[(.*?)\]", raw, re.DOTALL)
    if not match:
        return []
    items = [item.strip().strip('"').strip("'") for item in match.group(1).split(",")]
    return [item for item in items if item]


def parse_json(raw_text):
    # grab the reasoning trace, if the model produced one
    think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
    reasoning = think_match.group(1).strip() if think_match else None

    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

    if not isinstance(parsed, dict):
        return None

    if reasoning:
        parsed["reasoning"] = reasoning

    return parsed


# def parse_json(raw):
#     raw = raw.strip()
#     start = raw.find("{")
#     if start == -1:
#         return None
#     raw = raw[start:]

#     diff = raw.count("{") - raw.count("}")
#     if diff > 0:
#         raw += "}" * diff

#     try:
#         raw = re.sub(r',\s*([}\]])', r'\1', raw)
#         return json.loads(raw)
#     except json.JSONDecodeError as e:
#         print(e)
#         print(raw)
#         return None

## ------------------------------------------
# def parse_json(raw_text):
#     # Step 1: extract the reasoning trace first, if present
#     think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
#     reasoning = think_match.group(1).strip() if think_match else None

#     # Step 2: only look for JSON in whatever comes AFTER the </think> tag
#     after_think = raw_text.split("</think>", 1)[-1]

#     start = after_think.find("{")
#     if start == -1:
#         return None
#     candidate = after_think[start:]

#     # Step 3: repair truncated JSON by balancing braces (only within this slice)
#     diff = candidate.count("{") - candidate.count("}")
#     if diff > 0:
#         candidate += "}" * diff

#     try:
#         parsed = json.loads(candidate)
#     except json.JSONDecodeError as e:
#         print(e)
#         print(candidate)
#         return None

#     if not isinstance(parsed, dict):
#         return None

#     if reasoning:
#         parsed["reasoning"] = reasoning

#     return parsed

In [ ]:
## 6. Generation functions
  # max tokens allowed for the actual JSON/list answer, guaranteed separately

def generate_response_safe(full_prompt, think_budget=THINK_BUDGET, answer_budget=ANSWER_BUDGET):
    messages = [{"role": "user", "content": full_prompt}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    # --- Stage 1: generate up to think_budget tokens ---
    with torch.no_grad():
        think_ids = model.generate(
            **inputs,
            max_new_tokens=think_budget,
            do_sample=False, temperature=0.6, top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    think_text = tokenizer.decode(think_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # --- If the model didn't naturally close </think>, force it closed ---
    if "</think>" not in think_text:
        think_text = think_text + "\n</think>\n"

    # --- Stage 2: continue generating, fresh budget, just for the answer ---
    continued_prompt = prompt_text + think_text
    inputs2 = tokenizer(continued_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        answer_ids = model.generate(
            **inputs2,
            max_new_tokens=answer_budget,
            do_sample=False, temperature=0.6, top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer_text = tokenizer.decode(answer_ids[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True)

    del inputs, think_ids, inputs2, answer_ids
    torch.cuda.empty_cache()

    return think_text + answer_text  # full text, same shape parse_json already expects

def generate_community_response(brain_graph_text):
    prompt = build_community_prompt(brain_graph_text, roi_names, roi_to_network, top_k=TOP_K_PREDICTED)
    system_message = (
        "You are a neuroscience expert. Output only a bracketed list of network-pair names in the exact format requested (e.g. [Default-Default, Cont-DorsAttn]). No markdown, no extra commentary, no explanation outside the bracket list."
    )
    full_prompt = f"{system_message}\n\n{prompt}"
    return generate_response_safe(full_prompt, THINK_BUDGET, ANSWER_BUDGET)


def generate_classification_response(brain_graph_text, community_pairs=None):
    prompt = build_classification_prompt(brain_graph_text, community_pairs=community_pairs)
    system_message = (
        "You are a neuroscience expert. Output only one valid JSON object matching the required schema. No markdown, comments, or extra text. Must parse with json.loads()."
    )
    full_prompt = f"{system_message}\n\n{prompt}"
    return generate_response_safe(full_prompt, THINK_BUDGET, ANSWER_BUDGET)

In [ ]:
# ## 6. Generation functions
# def generate_response_safe(prompt, system_message, max_new_tokens, retries=RETRY_ATTEMPTS):
#     full_prompt = f"{system_message}\n\n{prompt}"
#     for attempt in range(retries + 1):
#         try:
#             response = client.chat.completions.create(
#                 model=MODEL_ID,
#                 messages=[{"role": "user", "content": full_prompt}],
#                 max_completion_tokens=max_new_tokens,
#                 reasoning_effort="low",   # add this
#             )
#             return response.choices[0].message.content
#         except Exception:
#             if attempt == retries:
#                 raise
#             time.sleep(2 * (attempt + 1))

##---------------------------------------------------------------------------
# def generate_response_safe(prompt, system_message, max_new_tokens, retries=RETRY_ATTEMPTS):
#     full_prompt = f"{system_message}\n\n{prompt}"
#     for attempt in range(retries + 1):
#         try:
#             response = client.chat.completions.create(
#                 model=MODEL_ID,
#                 messages=[{"role": "user", "content": full_prompt}],
#                 max_completion_tokens=max_new_tokens,   # unchanged value, just the correct param name for this API
#             )
#             return response.choices[0].message.content
#         except Exception:
#             if attempt == retries:
#                 raise
#             time.sleep(2 * (attempt + 1))



# def generate_community_response(brain_graph_text):
#     prompt = build_community_prompt(brain_graph_text, roi_names, roi_to_network, top_k=TOP_K_PREDICTED)
#     system_message = (
#         "You are a neuroscience expert. Output only a bracketed list of network-pair names in the exact format requested (e.g. [Default-Default, Cont-DorsAttn]). No markdown, no extra commentary, no explanation outside the bracket list."
#     )
#     return generate_response_safe(prompt, system_message, MAX_NEW_TOKENS_COMMUNITY)


# def generate_classification_response(brain_graph_text, community_pairs=None):
#     prompt = build_classification_prompt(brain_graph_text, community_pairs=community_pairs)
#     system_message = (
#         "You are a neuroscience expert. Output only one valid JSON object matching the required schema. No markdown, comments, or extra text. Must parse with json.loads()."
#     )
#     return generate_response_safe(prompt, system_message, MAX_NEW_TOKENS_CLS)

In [13]:
## 7. SANITY CHECK -- run both prompts on ONE subject, inspect input + output
#     Do this before launching the full 768-subject loop.
sanity_sid = subject_ids[0]
sanity_edge_text = brain_graph_texts[sanity_sid]

print("=" * 90)
print(f"SANITY CHECK -- Subject {sanity_sid}")
print("=" * 90)

# ---- Step 1: Community prompt -- input ----
community_prompt_preview = build_community_prompt(sanity_edge_text, roi_names, roi_to_network, top_k=TOP_K_PREDICTED)
print("\n--- [STEP 1] COMMUNITY PROMPT (INPUT) ---")
print(community_prompt_preview)
print(f"\n[community prompt tokens: {len(tokenizer(community_prompt_preview)['input_ids'])}]")

# ---- Step 1: Community prompt -- output ----
community_raw_sanity = generate_community_response(sanity_edge_text)
community_parsed_sanity = parse_community_list(community_raw_sanity)

print("\n--- [STEP 1] COMMUNITY RESPONSE (PARSED) ---")
print(community_parsed_sanity)

# ---- Step 2: Classification prompt -- input (chained with community output) ----
cls_prompt_preview = build_classification_prompt(sanity_edge_text, community_pairs=community_parsed_sanity)
print("\n" + "=" * 90)
print("--- [STEP 2] CLASSIFICATION PROMPT (INPUT, CHAINED WITH COMMUNITY OUTPUT) ---")
print(cls_prompt_preview)
print(f"\n[classification prompt tokens: {len(tokenizer(cls_prompt_preview)['input_ids'])}]")

# ---- Step 2: Classification prompt -- output ----
cls_raw_sanity = generate_classification_response(sanity_edge_text, community_pairs=community_parsed_sanity)
cls_parsed_sanity = parse_json(cls_raw_sanity)

print("\n--- [STEP 2] CLASSIFICATION RESPONSE (PARSED) ---")
print(cls_parsed_sanity)
print(cls_raw_sanity)

print("\n" + "=" * 90)
print("SANITY CHECK COMPLETE -- review both steps above before running full loop.")
print("=" * 90)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SANITY CHECK -- Subject 1018959

--- [STEP 1] COMMUNITY PROMPT (INPUT) ---
You are required to identify the most densely connected functional network pairs in the given brain functional connectivity network and output the top pairs ranked by connectivity.
Each region belongs to one of 8 functional groupings: the 7 Yeo networks (Vis, SomMot, DorsAttn, SalVentAttn, Limbic, Cont, Default) plus 'Unassigned' for regions with no cortical Yeo-7 overlap. A network pair (same network twice for within-network, or two different networks for between-network) is highly connected if its regions appear frequently together across the connection list. To identify the top pairs, count how many times each network pair appears across the connection list -- pairs appearing most frequently are the most connected.

**Example**
- Regions and their networks: Frontal_Sup_L (Cont), Frontal_Sup_R (Cont), Parietal_Inf_L (DorsAttn), Occipital_Mid_L (Vis), Temporal_Sup_R (SalVentAttn)
- Functional connections: Front

/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



--- [STEP 1] COMMUNITY RESPONSE (PARSED) ---
['Cont-Vis', 'Cont-SomMot', 'DorsAttn-Vis', 'Cont-Default', 'SalVentAttn-Vis', 'DorsAttn-SomMot', 'SalVentAttn-Default', 'Cont-SalVentAttn', 'DorsAttn-Default', 'SalVentAttn-SomMot']

--- [STEP 2] CLASSIFICATION PROMPT (INPUT, CHAINED WITH COMMUNITY OUTPUT) ---
Template: This data comes from the ADHD-200 dataset, preprocessed using the AAL116 brain atlas template (116 regions of interest).
Description: Functional connectivity edges for one subject from resting-state fMRI. Format: RegionA:weight:RegionB, identified by their anatomical region name (AAL atlas).
Task: Based on the edge list below and the identified network pairs, predict Control or ADHD with a confidence score for each class.
Request: Output only this JSON:
{
"prediction": "Control or ADHD",
"class_confidence": {
"Control": 0.0,
"ADHD": 0.0
},
"reasoning": "(1-2 sentence)"
}
Edges:
Temporal_Sup_L:0.92:Temporal_Sup_R
Rectus_L:0.90:Rectus_R
Cingulum_Ant_L:0.87:Cingulum_Ant_R
Fron

In [3]:
import os

OUTPUT_DIR = "/home/siu856622573/Code/RQ_4_5/deepseek/community"

CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "deepseek_community_for_classification_checkpoint_10com.json")

FINAL_PATH = os.path.join(OUTPUT_DIR, "deepseek_community_for_classification_10com.json")

LOG_PATH = os.path.join(OUTPUT_DIR, "deepseek_community_for_classification_errors_10com.log")

SAVE_EVERY = 10         
RETRY_ATTEMPTS = 2       


In [15]:
## 8. Checkpoint / logging utilities
def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            loaded = json.load(f)
        loaded = {
            sid: rec for sid, rec in loaded.items()
            if rec.get("prediction") is not None and rec.get("community_pairs")
        }
        print(f"Resuming: {len(loaded)} subjects already completed successfully.")
        return loaded
    return {}


def save_checkpoint(results, path=CHECKPOINT_PATH):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp_path, path)


def log_line(msg):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_PATH, "a") as f:
        f.write(f"[{ts}] {msg}\n")

In [ ]:
## 9. Inference loop -- runs both prompts independently per subject
from tqdm.auto import tqdm
import time

results = load_checkpoint()
remaining_ids = [sid for sid in subject_ids if sid not in results]
print(f"Total subjects: {len(subject_ids)} | already done: {len(results)} | remaining: {len(remaining_ids)}")

n_success, n_comm_parse_fail, n_cls_parse_fail, n_error = 0, 0, 0, 0
start_time = time.time()

with open(LOG_PATH, "a") as log_f:
    pbar = tqdm(remaining_ids, desc="Subjects", unit="subj")
    for idx, sid in enumerate(pbar):
        try:
            brain_graph_text = brain_graph_texts[sid]

            if meta is not None:
                gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
                ground_truth = "Control" if gt == 0 else "ADHD"
            else:
                ground_truth = None

            # -- step 1: Community-identification call --
            comm_raw = generate_community_response(brain_graph_text)
            comm_parsed = parse_community_list(comm_raw)
            comm_failed = not comm_parsed
            if comm_failed:
                n_comm_parse_fail += 1
                log_f.write(f"[community_parse_fail] {sid}: {comm_raw[:300]!r}\n")

            # -- step 2: classification call, CHAINED with community output --
            cls_raw = generate_classification_response(brain_graph_text, community_pairs=comm_parsed)
            cls_parsed = parse_json(cls_raw)
            if not cls_parsed:
                n_cls_parse_fail += 1
                log_f.write(f"[cls_parse_fail] {sid}: {cls_raw[:300]!r}\n")

            results[sid] = {
                "subject_id": sid,
                "ground_truth": ground_truth,

                # community-identification: parsed list if successful, else raw text
                "community_pairs": comm_parsed if not comm_failed else None,
                "community_raw_output": comm_raw if comm_failed else None,

                # classification: parsed fields if successful, else raw text
                "prediction": cls_parsed.get("prediction") if cls_parsed else None,
                "class_confidence": cls_parsed.get("class_confidence") if cls_parsed else None,
                "reasoning": cls_parsed.get("reasoning") if cls_parsed else None,
                "cls_raw_output": cls_raw if not cls_parsed else None,
            }
            n_success += 1
            log_f.flush()

        except Exception as e:
            n_error += 1
            log_f.write(f"[error] {sid}: {repr(e)}\n")
            log_f.flush()
            continue

        if (idx + 1) % SAVE_EVERY == 0:
            save_checkpoint(results)

save_checkpoint(results)
elapsed = time.time() - start_time
print(f"\nDone in {elapsed/60:.1f} min | success: {n_success} | "
      f"community parse fail: {n_comm_parse_fail} | cls parse fail: {n_cls_parse_fail} | errors: {n_error}")

if len(results) == len(subject_ids): 
    os.replace(CHECKPOINT_PATH, FINAL_PATH)
    print(f"All subjects done -> {FINAL_PATH}")
else:
    print(f"Not all subjects done yet -- re-run this cell to resume from {CHECKPOINT_PATH}")

In [4]:
import json, os

VALID_LABELS = {"ADHD", "Control"}
# FINAL_PATH = "llama_all_edge_list_pos10_identify_hub_4_checkpoint.json"
# FINAL_PATH = "llama_all_edge_list_pos10_identify_hub_2.json"

def print_stats(data):
    n = len(data)

    n_no_pred = sum(1 for r in data.values() if r.get("prediction") is None)
    malformed_entries = [
        (sid, r.get("prediction"))
        for sid, r in data.items()
        if r.get("prediction") is not None and r.get("prediction") not in VALID_LABELS
    ]
    n_malformed_pred = len(malformed_entries)

    # scorable = BOTH ground_truth and prediction are exactly "ADHD" or "Control"
    scorable = [
        r for r in data.values()
        if r.get("ground_truth") in VALID_LABELS and r.get("prediction") in VALID_LABELS
    ]
    n_scorable = len(scorable)

    n_gt_adhd_all = sum(1 for r in data.values() if r.get("ground_truth") == "ADHD")
    n_gt_control_all = sum(1 for r in data.values() if r.get("ground_truth") == "Control")

    n_pred_adhd = sum(1 for r in scorable if r["prediction"] == "ADHD")
    n_pred_control = sum(1 for r in scorable if r["prediction"] == "Control")

    n_correct = sum(1 for r in scorable if r["prediction"] == r["ground_truth"])

    # Per-class, denominator restricted to scorable subjects only
    scorable_gt_adhd = [r for r in scorable if r["ground_truth"] == "ADHD"]
    scorable_gt_control = [r for r in scorable if r["ground_truth"] == "Control"]
    n_adhd_correct = sum(1 for r in scorable_gt_adhd if r["prediction"] == "ADHD")
    n_control_correct = sum(1 for r in scorable_gt_control if r["prediction"] == "Control")

    print(f"Total subjects in file:              {n}")
    print(f"  -- no prediction (parse failure):  {n_no_pred}")
    print(f"  -- malformed prediction (neither ADHD nor Control exactly): {n_malformed_pred}")
    if malformed_entries:
        print(f"     malformed subject_ids and their prediction values:")
        for sid, val in malformed_entries:
            print(f"       {sid}: {val!r}")
    print(f"  -- scorable (valid ground_truth AND valid prediction): {n_scorable}")
    print()
    print(f"Ground truth (all {n} subjects) -- ADHD: {n_gt_adhd_all} | Control: {n_gt_control_all}")
    print(f"Prediction (scorable subjects only, n={n_scorable}) -- ADHD: {n_pred_adhd} | Control: {n_pred_control}")
    print()
    if n_scorable:
        print(f"Accuracy (scorable only): {n_correct}/{n_scorable} = {n_correct / n_scorable:.2%}")
    else:
        print("Accuracy: n/a (no scorable subjects)")

    print()
    print("Correctly predicted, by class (denominator = scorable subjects of that class only):")
    if scorable_gt_adhd:
        print(f"  ADHD    correctly predicted: {n_adhd_correct}/{len(scorable_gt_adhd)} = {n_adhd_correct / len(scorable_gt_adhd):.2%}")
    else:
        print("  ADHD    correctly predicted: n/a")
    if scorable_gt_control:
        print(f"  Control correctly predicted: {n_control_correct}/{len(scorable_gt_control)} = {n_control_correct / len(scorable_gt_control):.2%}")
    else:
        print("  Control correctly predicted: n/a")


if os.path.exists(FINAL_PATH):
    with open(FINAL_PATH) as f:
        data = json.load(f)
    print("RUN COMPLETE.")
    print_stats(data)


else:
    print("No output file found yet -- nothing processed.")

print(FINAL_PATH)

RUN COMPLETE.
Total subjects in file:              768
  -- no prediction (parse failure):  0
  -- malformed prediction (neither ADHD nor Control exactly): 0
  -- scorable (valid ground_truth AND valid prediction): 768

Ground truth (all 768 subjects) -- ADHD: 280 | Control: 488
Prediction (scorable subjects only, n=768) -- ADHD: 33 | Control: 735

Accuracy (scorable only): 477/768 = 62.11%

Correctly predicted, by class (denominator = scorable subjects of that class only):
  ADHD    correctly predicted: 11/280 = 3.93%
  Control correctly predicted: 466/488 = 95.49%
/home/siu856622573/Code/RQ_4_5/deepseek/community/deepseek_community_for_classification_10com.json
